<a href="https://colab.research.google.com/github/lalawee/special-octo-barnacle/blob/main/GridWorld_MC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Monte Carlo First Visit Method to solve a simple GridWorld problem.#
Goal is to estimate the state-value function V(s) for a given policy.
<br>In this case, we are using a RANDOM UNIFORM POLICY to train.

In [10]:
import numpy as np

# Define Environment Class
class GridWorld:
  def __init__(self, row, col, start):
    self.row = row
    self.col = col
    self.i = start[0]
    self.j = start[1]
    self.start_row = start[0]
    self.start_col = start[1]

  def set(self, rewards, actions, terminal_states):
    # rewards should be a dict of: (i, j): r (row, col): reward
    # actions should be a dict of: (i, j): A (row, col): list of possible actions
    self.rewards = rewards #dict
    self.actions = actions #dict
    self.terminal_states = terminal_states #list

  def reset(self):
    self.i = self.start_row
    self.j = self.start_col
    return (self.start_row, self.start_col)

  #
  # next_state, reward, done, invalid = env.step(action)
  #
  def step(self, action):
    # check if legal move first
    if action in self.actions[(self.i, self.j)]: #if action in tuple of permitted actions
      if action == 'U': self.i -= 1
      elif action == 'D': self.i += 1
      elif action == 'R': self.j += 1
      elif action == 'L': self.j -= 1
    # invalid move
    else:
      return (0,0), 0, False, True

    next_state = (self.i, self.j)
    reward = self.rewards.get(next_state, 0)
    invalid = False
    done = True if next_state in self.terminal_states else False

    return next_state, reward, done, invalid

  # Example:
  # return
  # {(0, 1), (1, 2), (2, 1), (0, 2), (2, 2), (1, 0), (1, 3), (0, 0), (1, 1), (0, 3), (2, 0), (2, 3)}
  #
  def all_states(self):
    states=set()
    for i in range(self.row):
      for j in range(self.col):
        states.add((i,j))
    return states # return a set of states, each state in tuple form

#
# ============================================================================================
#
def standard_grid():
  # define a grid that describes the reward for arriving at each state
  # and possible actions at each state
  # the grid looks like this
  # x means you can't go there
  # s means start position
  # number means reward at that state
  #   0  1  2  3
  # 0 .  .  .  1
  # 1 .  x  . -1
  # 2 s  .  .  .
  #
  # We go by (row, col)
  #
  g = GridWorld(3, 4, (2, 0)) #row, col, start pos

  rewards = {
    (0, 0):  0,
    (0, 1):  0,
    (0, 2):  0,
    (0, 3):  1,
    (1, 0):  0,
    (1, 1):  0,
    (1, 2):  0,
    (1, 3): -1,
    (2, 0):  0,
    (2, 1):  0,
    (2, 2):  0,
    (2, 3):  0
  }

  actions = {
    (0, 0): ('D', 'R'),
    (0, 1): ('L', 'R'),
    (0, 2): ('L', 'D', 'R'),
#   (0, 3): Terminal
    (1, 0): ('U', 'D'),
#   (1, 1): Blocked
    (1, 2): ('U', 'D', 'R'),
#   (1, 3): Terminal
    (2, 0): ('U', 'R'),
    (2, 1): ('L', 'R'),
    (2, 2): ('L', 'R', 'U'),
    (2, 3): ('L', 'U')
  }

  terminal_states = [(0,3),(1,3)]

  g.set(rewards, actions, terminal_states)
  return g #return GridWorld object

#
# Print values given V dict and g grid
#
def print_values(V, g):
  for i in range(g.row):
    print("---------------------------")
    for j in range(g.col):
      v = V.get((i,j), 0) # V is a state value dict
      if v >= 0:
        print(" %.2f|" % v, end="")
      else:
        print("%.2f|" % v, end="") # -ve sign takes up an extra space
    print("")

#
# policy means mapping
# at each state, what should agent do
# P input is a dict, e.g. {(0,0):'U', (0,1):'L', ... }
#
def print_policy(P, g):
  for i in range(g.row):
    print("---------------------------")
    for j in range(g.col):
      a = P.get((i,j), ' ')
      print("  %s  |" % a, end="")
    print("")

#**Main starts here**

**Standard Grid:**<br>
x means you can't go there<br>
s means start position<br>
number means reward at that state<br>
**.  .  .  1**<br>
**.  x  . -1**<br>
**s  .  .  .**<br>





In [11]:
# define a grid that describes the reward for arriving at each state and possible actions at each state
# the grid looks like this:
#
# x means you can't go there
# s means start position
# number means reward at that state
#   0  1  2  3
# 0 .  .  .  1
# 1 .  x  . -1
# 2 s  .  .  .
#
# We go by (row, col)
#
MAX_EPISODES = 10000
GAMMA = 0.9
CHECKPOINTS = [100, 500, 1000, 5000, 10000]
env = standard_grid() #3x4 grid
returns = {}  # Dictionary to store returns for each state, returns[s] points to a list
state_values = {} # Dictionary to store state value functions

In [12]:
#
# Generate Episodes
#
for ep in range(MAX_EPISODES):
	state = env.reset() #return a tuple, state at start
	trajectory = [] #to store trajectory of 1 episode

  #
	# generate 1 episode
  #
	while True:
		# using UNIFORM RANDOM as a policy to train (behavioral policy)
		action = np.random.choice(['U','D','L','R'])

		next_state, reward, done, invalid = env.step(action)
		if invalid: continue #go back and get another random action

		trajectory.append((state, reward)) #St, Rt+1
		state = next_state
		if done: break

  #
	# Analyse the 1 episode generated
	# E.g. trajectory = [((2, 0), 0), ((2, 1), 0), ((2, 0), 0), ((2, 1), 0), ((2, 2), 0), ((1, 2), -1)]
	#
	reverse_trajectory = trajectory[::-1]
	reverse_trajectory_states = list(map(lambda x:x[0], reverse_trajectory)) # just the states only
	G = 0

	#
	# GO THRU trajectory BACKWARD to CALC averages
	# trajectory[] contains (state, reward) of only 1 episode
	#
	for idx, step in enumerate(reverse_trajectory):
		s, r = step
		G = GAMMA*G + r # say, after 3 steps, G=GAMMA*(GAMMA*(GAMMA*G + r3) + r2) + r1
                    #                     G=GAMMA^3*G + GAMMA^2*r3 + GAMMA*r2 + r1
                    #                     G=GAMMA^2*r3 + GAMMA*r2 + r1

    #
    # First-visit MC: only update the FIRST OCCURENCE of the state
    #                         5       4       3       2       1       0
		# E.g. for trajectory = [(2, 0), (2, 1), (2, 0), (2, 1), (2, 2), (1, 2)]
		#      it will be idx: 0, 1, 4, 5
		#      Cause for example when idx=2, state is (2,1), but this state is in reverse_trajectory_states[3:],
		#      reverse_trajectory_states[3:] means [(2,0),(2,1),(2,0)], so we will ignore its processing.
		#      Hence, we only consider the state (2,1) from the first occurence counting from the start of trajectory.
		#
		if s not in reverse_trajectory_states[idx+1:]:
			# we add state s to dict returns, returns[s] points to a list
			if s in returns: returns[s].append(G) # returns is a dict that points to lists
			else:            returns[s] = [G] # first guy in the list

			# update state_values[s] dict, which stores mean gain results of each state s
			state_values[s] = np.mean(returns[s]) # returns[s] points to a list

    # The terminal states will have 0 value cause no further rewards are received after reaching them.
    # That is, agent doesn't move from terminal states to get further rewards. That is why we
    # manually insert the 2 final values (rewards) in the terminal states in the state_values dict.
		state_values[(0,3)]=1
		state_values[(1,3)]=-1
	if (ep + 1) in CHECKPOINTS:
			print(f"\n=== State Values after {ep+1} episodes ===")
			print_values(state_values, env)
# print("Training complete.")


=== State Values after 100 episodes ===
---------------------------
 0.13| 0.24| 0.29| 1.00|
---------------------------
-0.03| 0.00|-0.37|-1.00|
---------------------------
-0.11|-0.26|-0.38|-0.71|

=== State Values after 500 episodes ===
---------------------------
 0.08| 0.18| 0.31| 1.00|
---------------------------
-0.03| 0.00|-0.37|-1.00|
---------------------------
-0.11|-0.22|-0.39|-0.66|

=== State Values after 1000 episodes ===
---------------------------
 0.06| 0.15| 0.29| 1.00|
---------------------------
-0.02| 0.00|-0.34|-1.00|
---------------------------
-0.11|-0.22|-0.37|-0.68|

=== State Values after 5000 episodes ===
---------------------------
 0.05| 0.15| 0.27| 1.00|
---------------------------
-0.02| 0.00|-0.36|-1.00|
---------------------------
-0.11|-0.22|-0.37|-0.67|

=== State Values after 10000 episodes ===
---------------------------
 0.05| 0.14| 0.27| 1.00|
---------------------------
-0.02| 0.00|-0.37|-1.00|
---------------------------
-0.11|-0.22|-0.37|-0.

###Note: This is an ON POLICY RL. In the example, the random policy is both the behavior policy (used to generate episodes) and the target policy (being evaluated or updated). There is no separation between the behavior policy and the target policy. The values V(s) represent the expected return when following say, random policy and the value estimates are directly tied to the same random policy being followed. We didn't further alter the values by weighting them during updates, for example.
